In [2]:
import pandas as pd
import my_regression


In [3]:
dataset=pd.read_csv("insurance_pre.csv")
dataset = pd.get_dummies(dataset, drop_first=True)

X = dataset.drop("charges", axis=1).values
y = dataset["charges"].values


In [4]:
# Used Linear Regression with Cross-Validation
lin_reg_model, linreg_mean_r2_score = my_regression.lin_reg(X, y)


No. of folds: 5
R² Scores for each fold:    R2_Score
0  0.693023
1  0.713482
2  0.783058
3  0.763651
4  0.751350
Average R² Score: 0.7409129106560753


In [5]:
# Using SVM Regression with Grid Search CV

param_grid = {'kernel':['linear', 'poly', 'rbf', 'sigmoid'], 'C':[50, 100, 500, 1000, 2000, 3000],'gamma':['auto','scale']} 
# scoring_method = 'r2'

svr_reg_model, st_scaler = my_regression.regress('SVR', X, y, param_grid, n_folds=10,  scoring_method='r2')


Model Selected : SVR
Scaling the data using StandardScaler...
Fitting 10 folds for each of 48 candidates, totalling 480 fits
   param_kernel param_C param_gamma  mean_test_score  std_test_score  \
0           rbf    3000        auto         0.829278        0.071207   
1           rbf    3000       scale         0.829276        0.071171   
2           rbf    2000        auto         0.822628        0.070317   
3           rbf    2000       scale         0.822614        0.070291   
4          poly    2000        auto         0.821612        0.068566   
5          poly    2000       scale         0.821587        0.068514   
6          poly    3000        auto         0.821557        0.068911   
7          poly    3000       scale         0.821540        0.068880   
8          poly    1000       scale         0.821129        0.067535   
9          poly    1000        auto         0.821102        0.067511   
10         poly     500       scale         0.804894        0.066002   
11         

In [6]:
# Using DecisionTree Regression with Grid Search CV

param_grid = {'criterion':['squared_error','absolute_error','friedman_mse', 'poisson'],
              'max_features': [None,'sqrt','log2'],
              'splitter':['best','random']} 

dt_reg_model, _ = my_regression.regress('DecisionTreeRegressor',X, y, param_grid, n_folds=10,  scoring_method='r2')


Model Selected : DecisionTreeRegressor
Model does not require scaling.
Fitting 10 folds for each of 24 candidates, totalling 240 fits
   param_criterion param_max_features param_splitter  mean_test_score  \
0          poisson               None           best         0.707749   
1   absolute_error               sqrt           best         0.700896   
2   absolute_error               None         random         0.700100   
3    squared_error               sqrt           best         0.694233   
4          poisson               None         random         0.684862   
5          poisson               log2           best         0.684185   
6          poisson               sqrt           best         0.682304   
7   absolute_error               None           best         0.681180   
8    squared_error               None         random         0.678146   
9     friedman_mse               None           best         0.677640   
10  absolute_error               sqrt         random         0.

In [7]:
# Using RandomForest Regression with Grid Search CV

param_grid = {'criterion':['absolute_error', 'friedman_mse', 'poisson', 'squared_error'],
              'max_features': [None,'sqrt','log2'],
              'n_estimators':[10,50,100]} 
rf_reg_model, _ = my_regression.regress('RandomForestRegressor',X, y, param_grid, n_folds=10,  scoring_method='r2')


Model Selected : RandomForestRegressor
Model does not require scaling.
Fitting 10 folds for each of 36 candidates, totalling 360 fits
   param_criterion param_max_features param_n_estimators  mean_test_score  \
0          poisson               sqrt                100         0.833082   
1   absolute_error               sqrt                 50         0.833055   
2   absolute_error               log2                100         0.832699   
3    squared_error               sqrt                 50         0.832685   
4    squared_error               sqrt                100         0.832329   
5    squared_error               log2                100         0.832302   
6   absolute_error               sqrt                100         0.832161   
7   absolute_error               log2                 50         0.831999   
8     friedman_mse               log2                100         0.831412   
9          poisson               sqrt                 50         0.831145   
10         poisson 

In [ ]:
# Find Best Model with Best R2 Score
all_models_score = {
    'Linear Regression': [linreg_mean_r2_score, lin_reg_model], 
    'SVR': [svr_reg_model.best_score_,svr_reg_model],
    'Decision Tree': [dt_reg_model.best_score_, dt_reg_model],
    'Random Forest': [rf_reg_model.best_score_, rf_reg_model]
}

best_model_name = None
best_r2_score = 0.00  
best_model = None

for model_name, (r2_score, model) in all_models_score.items():
    if r2_score > best_r2_score:
        best_r2_score = r2_score
        best_model = model
        best_model_name = model_name

print(f"Best Model: {best_model_name} with R2 Score: {best_r2_score}")



Best Model: Random Forest with R2 Score: 0.8330817887858835


In [24]:
import pickle
filename="finalized_model_regression.sav"
pickle.dump(best_model,open(filename,'wb'))

loaded_model=pickle.load(open("finalized_model_regression.sav",'rb'))
# age:45,bmi:28,children:2,sex_male:1,smoker_yes:0
result=loaded_model.predict([[45,28,2,1,0]])
print('Insurance Charges : Rs.',result[0])


Insurance Charges : Rs. 9649.5574367
